# High-rise 06 - Structure prints

Report-ready renders of the building model itself: plain geometry from the
standard views, the facade partition (by outward normal), and the floor
partition (by centroid height against the case floor edges). These document
the model that produced the coefficients and are written to `deliverables/`
(geometry) and `debug/` (partitions).

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

import matplotlib

matplotlib.use("Agg")  # headless: notebooks write files, they do not display

from cfdmod import inflow_report, mesh_field, plot_config  # noqa: E402
from cfdmod.building import (  # noqa: E402
    BuildingCase,
    cf_per_floor,
    cm_per_floor,
    cp_from_pressure,
    example_building_case,
    example_building_structure,
    floor_accelerations,
    floor_load_source,
    peak_response_table,
    peak_value,
    plot_floor_mass,
    plot_mode_shape,
    plot_natural_frequencies,
    solve_building_response,
    structure_from_csvs,
)
from cfdmod.report import DebugWriter  # noqa: E402


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plot_config.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
# --- config -------------------------------------------------------------
MESH = pathlib.Path(
    os.environ.get("CFDMOD_HR_MESH", FIX / "pressure" / "galpao" / "galpao.normalized.lnas")
)
CASE_DATA = os.environ.get("CFDMOD_HR_CASE_DATA")
PARAMS = os.environ.get("CFDMOD_HR_PARAMS", "params_cat3.yaml")
if CASE_DATA:
    case = BuildingCase.from_case_data(pathlib.Path(CASE_DATA), PARAMS)
else:
    case = example_building_case(MESH)

geom = mesh_field.load_geometry(MESH)
n_tri = int(np.asarray(geom.triangle_vertices).shape[0])

# Per-triangle floor index from centroid height vs the case z-edges.
tri = np.asarray(geom.triangle_vertices)
cz = tri[:, :, 2].mean(axis=1)
edges = np.asarray(case.floor_heights)
floor_id = np.clip(np.digitize(cz, edges[1:-1]), 0, len(edges) - 2).astype(float)
groups = mesh_field.facade_groups(MESH)
fac_idx = mesh_field.facade_index_per_triangle(groups, n_tri)
print(f"{n_tri} triangles | {case.n_floors} floors | {len(groups)} facade groups")

In [ ]:
# --- render -------------------------------------------------------------
dbg = DebugWriter(OUTPUT_BASE, stage="structure", version=VERSION)

for v in ("iso", "front", "right", "top"):
    fig, _ = mesh_field.triangle_field_figure(
        geom, None, view=mesh_field.STANDARD_VIEWS[v], title=f"geometry - {v}"
    )
    dbg.savefig(fig, f"geometry_{v}.png", deliverable=True)
    plot_config.close(fig)

fig, _ = mesh_field.triangle_field_figure(
    geom, fac_idx, cmap="tab10", view=mesh_field.STANDARD_VIEWS["iso"],
    title="facade partition", cbar_label="facade id",
)
dbg.savefig(fig, "facade_partition.png")
plot_config.close(fig)

fig, _ = mesh_field.triangle_field_figure(
    geom, floor_id, cmap="viridis", view=mesh_field.STANDARD_VIEWS["iso"],
    title="floor partition", cbar_label="floor",
)
dbg.savefig(fig, "floor_partition.png")
plot_config.close(fig)
print("structure prints under", dbg.deliverables_dir)